In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  16.4M      0 --:--:-- --:--:-- --:--:-- 16.4M


In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 9.9 MB/s eta 0:00:00


In [4]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml

from IPython.display import Video, display
from ultralytics import YOLO
from ultralytics.utils import ROOT

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
DAY_02_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02"
)

RAW_VIDEO_PATH = Path("/content/development.mp4")
DEFAULT_TRACKS_PATH = DAY_02_DIR / "tracks.csv"
DEFAULT_VIDEO_PATH = DAY_02_DIR / "bytetrack_baseline.mp4"

DAY_04_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_04"
)

DAY_04_DIR.mkdir(parents=True,exist_ok=True)

BUFFER60_VIDEO_PATH = DAY_04_DIR / "bytetrack_buffer60.mp4"
BUFFER60_TRACKS_PATH = DAY_04_DIR / "tracks_buffer60.csv"
CUSTOM_TRACKER_PATH = DAY_04_DIR / "bytetrack_buffer60.yaml"

MODEL_NAME = "yolo26n.pt"
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.10
NMS_IOU_THRESHOLD = 0.70

TARGET_CLASS_NAMES = {
    "person",
    "bicycle",
    "car",
}

DEVICE = (0 if torch.cuda.is_available() else "cpu")

In [ ]:
def read_video_metadata(path):
    capture = cv2.VideoCapture(str(path))

    metadata = {
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "frames": int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    }

    capture.release()
    return metadata


raw_metadata = read_video_metadata(RAW_VIDEO_PATH)
default_video_metadata = read_video_metadata(DEFAULT_VIDEO_PATH)

metadata_comparison_df = pd.DataFrame(
    [
        {
            "source": "raw_video",
            **raw_metadata,
        },
        {
            "source": "day2_baseline",
            **default_video_metadata,
        },
    ]
)

display(metadata_comparison_df)


same_video_structure = (
    raw_metadata["width"] == default_video_metadata["width"]
    and
    raw_metadata["height"] == default_video_metadata["height"]
    and
    raw_metadata["frames"] == default_video_metadata["frames"]
    and
    abs(raw_metadata["fps"] - default_video_metadata["fps"])
    < 0.1
)

,source,width,height,fps,frames
0,raw_video,768,432,12.0,647
1,day2_baseline,768,432,12.0,647


**Custom ByteTrack configuration**

In [8]:
DEFAULT_TRACKER_PATH = Path(ROOT) / "cfg" / "trackers" / "bytetrack.yaml"

with open(DEFAULT_TRACKER_PATH, "r", encoding="utf-8") as file:
    default_tracker_config = yaml.safe_load(file)
    

buffer60_tracker_config = default_tracker_config.copy()
buffer60_tracker_config["track_buffer"] = 60

with open(CUSTOM_TRACKER_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(buffer60_tracker_config, file, sort_keys=False
    )

In [ ]:
changed_keys = sorted(
    key for key in set(default_tracker_config) | set(buffer60_tracker_config)
    if (default_tracker_config.get(key) != buffer60_tracker_config.get(key))
)

config_comparison_df = pd.DataFrame(
    [
        {
            "setting": key,
            "default": default_tracker_config.get(key),
            "buffer60":buffer60_tracker_config.get(key)
        }
        for key in default_tracker_config
    ]
)

display(config_comparison_df)
print("Changed keys:", changed_keys)

,setting,default,buffer60
0,tracker_type,bytetrack,bytetrack
1,track_high_thresh,0.25,0.25
2,track_low_thresh,0.1,0.1
3,new_track_thresh,0.25,0.25
4,track_buffer,30,60
5,match_thresh,0.8,0.8
6,fuse_score,True,True


Changed keys: ['track_buffer']


**Increased-buffer tracking run**

In [14]:
def run_tracking_experiment(video_path, tracker_config_path, output_video_path, output_csv_path):
    model = YOLO(MODEL_NAME)
    
    if isinstance(model.names, dict):
        class_id_to_name = {
            int(class_id) : str(class_name)
            for class_id, class_name in model.names.items()
        }
    else:
        class_id_to_name = {
            class_id: str(class_name)
            for class_id, class_name in enumerate(model.names)
        }
    
    target_class_id = sorted(
        class_id for class_id, class_name in class_id_to_name.items()
        if class_name.lower() in TARGET_CLASS_NAMES
    )
    
    capture = cv2.VideoCapture(str(video_path))
    
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    source_fps = float(capture.get(cv2.CAP_PROP_FPS))
    expected_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    
    writer = cv2.VideoWriter(
        str(output_video_path), 
        cv2.VideoWriter_fourcc(*"mp4v"),
        source_fps, (width, height)
    )
    
    columns = [
        "frame_index",
        "timestamp_seconds",
        "track_id",
        "class_id",
        "class_name",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
        "center_x",
        "center_y",
    ]
    rows = []
    processed_frames = 0
    
    try:
        while True:
            success, frame = capture.read()
            if not success:
                break
            
            frame_index = processed_frames
            result = model.track(
                source=frame,
                tracker=str(tracker_config_path),
                classes=target_class_id,
                conf=CONFIDENCE_THRESHOLD,
                iou=NMS_IOU_THRESHOLD,
                imgsz=IMAGE_SIZE,
                device=DEVICE,
                verbose=False
            )[0]
            
            boxes = result.boxes
            has_track_ids = (
                boxes is not None
                and len(boxes) > 0 and boxes.id is not None
            )
            
            if has_track_ids:
                xyxy_values = boxes.xyxy.detach().cpu().numpy()
                
                confidence_values = boxes.conf.detach().cpu().numpy()

                class_ids = boxes.cls.detach().cpu().numpy().astype(int)
                
                track_ids = boxes.id .detach().cpu().numpy().astype(int)
                
                for (xyxy, confidence, class_id, track_id) in zip(
                        xyxy_values, confidence_values,
                        class_ids, track_ids
                    ):
                        x1, y1, x2, y2 = map(float, xyxy)
                        
                        rows.append({
                            "frame_index": frame_index,
                            "timestamp_seconds": frame_index / source_fps,
                            "track_id": int(track_id),
                            "class_id": int(class_id),
                            "class_name": class_id_to_name[int(class_id)],
                            "confidence": float(confidence),
                            "x1": x1,
                            "y1": y1,
                            "x2": x2,
                            "y2": y2,
                            "center_x": (x1 + x2) / 2,
                            "center_y": (y1 + y2) / 2,
                    })
            
            annotated_frame = result.plot()
            writer.write(annotated_frame)
            
            processed_frames += 1
            if processed_frames % 100 == 0:
                print(
                    f"Processed "
                    f"{processed_frames}/"
                    f"{expected_frames}"
                )

    finally:
        capture.release()
        writer.release()
    
    output_df = pd.DataFrame(rows, columns=columns)
    output_df.to_csv(output_csv_path, index=False)
    
    return {
        "tracks_df": output_df,
        "processed_frames": processed_frames,
        "expected_frames": expected_frames,
        "source_fps": source_fps,
        "target_class_ids": target_class_id
    }

In [15]:
buffer60_result = (
    run_tracking_experiment(
        video_path = RAW_VIDEO_PATH,
        tracker_config_path = CUSTOM_TRACKER_PATH,
        output_video_path = BUFFER60_VIDEO_PATH,
        output_csv_path = BUFFER60_TRACKS_PATH
    )
)

buffer60_tracks_df = buffer60_result["tracks_df"]
default_tracks_df = pd.read_csv(DEFAULT_TRACKS_PATH)

print("Default rows:", len(default_tracks_df))
print("Buffer-60 rows:", len(buffer60_tracks_df))
print("Processed frames:", buffer60_result["processed_frames"])

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 339ms
Prepared 1 package in 47ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 100/647
Processed 200/647
Processed 300/647
Processed 400/647
Processed 500/647
Processed 600/647
Default rows: 306
Buffer-60 rows: 338
Processed frames: 647


## Diagnostic metrics

**Per-track analysis**

In [16]:
def analyze_track_table(tracks_df, source_fps):
    records = []
    short_track_limit = int(round(source_fps))

    for track_id, group in (
        tracks_df.sort_values(
            ["track_id", "frame_index"]
        ).groupby("track_id")
    ):
        frames = np.sort(group["frame_index"].unique())
        frame_differences = np.diff(frames)

        first_frame = int(frames[0])
        last_frame = int(frames[-1])

        observations = int(len(frames))

        span_frames = last_frame - first_frame + 1
        missing_inside_span = span_frames - observations
        
        gap_events = int(np.sum(frame_differences > 1))

        ordered_classes = group["class_name"].astype(str).to_numpy()

        class_changes = int(
            np.sum(ordered_classes[1:] != ordered_classes[:-1])
        )

        dominant_class = group["class_name"].mode().iloc[0]
        flags = []

        if observations <= short_track_limit:
            flags.append("short_track")

        if gap_events > 0:
            flags.append("internal_gap")

        if class_changes > 0:
            flags.append("class_flicker")

        coverage_ratio = observations / span_frames

        if coverage_ratio < 0.90:
            flags.append("low_coverage")

        mean_confidence = float(group["confidence"].mean())

        if mean_confidence < 0.35:
            flags.append("low_confidence")

        records.append(
            {
                "track_id": int(track_id),
                "dominant_class": dominant_class,
                "first_frame": first_frame,
                "last_frame": last_frame,
                "observations": observations,
                "span_frames": span_frames,
                "duration_seconds": span_frames / source_fps,
                "missing_inside_span": missing_inside_span,
                "gap_events": gap_events,
                "coverage_ratio": coverage_ratio,
                "unique_classes": int(group["class_name"].nunique()),
                "class_changes": class_changes,
                "mean_confidence": mean_confidence,
                "short_track": observations <= short_track_limit,
                "failure_flags": (
                    ", ".join(flags) if flags else "none"
                ),
            }
        )

    per_track_df = pd.DataFrame(records)

    active_tracks_per_frame = (
        tracks_df.groupby("frame_index")["track_id"].nunique()
    )

    summary = {
        "track_observations": len(tracks_df),
        "unique_track_ids": tracks_df["track_id"].nunique(),
        "mean_track_duration_s": per_track_df[
            "duration_seconds"
        ].mean(),
        "median_track_duration_s": per_track_df[
            "duration_seconds"
        ].median(),
        "short_tracks_1s_or_less": int(
            per_track_df["short_track"].sum()
        ),
        "tracks_with_gaps": int(
            (per_track_df["gap_events"] > 0).sum()
        ),
        "total_gap_events": int(
            per_track_df["gap_events"].sum()
        ),
        "missing_frames_inside_spans": int(
            per_track_df["missing_inside_span"].sum()
        ),
        "class_flicker_tracks": int(
            (per_track_df["class_changes"] > 0).sum()
        ),
        "mean_coverage_ratio": per_track_df[
            "coverage_ratio"
        ].mean(),
        "max_simultaneous_tracks": int(
            active_tracks_per_frame.max()
        ),
    }

    return per_track_df, summary
    

In [17]:
source_fps = raw_metadata["fps"]

default_per_track_df, default_summary = analyze_track_table(
    default_tracks_df, source_fps
)

buffer60_per_track_df, buffer60_summary= analyze_track_table(
    buffer60_tracks_df, source_fps
)

In [18]:
comparison_df = pd.DataFrame(
    [
        {
            "experiment": "buffer_30_default",
            **default_summary
        },
        {
            "experiment": "buffer_60",
            **buffer60_summary
        }
    ]
).set_index("experiment")

display(comparison_df.round(3))

,track_observations,unique_track_ids,mean_track_duration_s,median_track_duration_s,short_tracks_1s_or_less,tracks_with_gaps,total_gap_events,missing_frames_inside_spans,class_flicker_tracks,mean_coverage_ratio,max_simultaneous_tracks
experiment,,,,,,,,,,,
buffer_30_default,306,10,2.725,1.792,2,6,7,21,1,0.906,3
buffer_60,338,4,23.854,23.583,1,3,28,807,3,0.499,4
